# MobileViT

Imagine we want AI model to look at photo and answer:

> "is this a cat?"

There are two major ways we can build the model:

* CNN - good at looking locally
A CNN looks at small areas of image.

For example:

```
[small area] → edges
[small area] → eyes
[small area] → ears
[small area] → textures
```

CNNs are:
* Fast
* Small
* Good for mobile devices
* Good at detecting local patterns


BUt there is a problem;
A CNN doesnot naturally understand the whole image at once.

For example:
```
🐱
```
It might recognize: I see two eyes

and

I see two ears

but understanding : "These eyes + these ears + this body belongs to one Cat"

requires connecting information from different parts of image.


2. Transformers solve the "whole Image" problem

They are good at understanding relationships between distant things.

For example:

```
cat's ear ---------------- cat's tail
       \                    /
        \                  /
          → same object
```
A transformer can ask:

"how is this part of the image related to all other parts?"

This is global information or global representation.

so we have:



| Feature / Characteristic | Convolutional Neural Networks (CNN) | Transformers |
| --- | --- | --- |
| **Information Scope** | Great at local information | Great at global information |
| **Speed & Efficiency** | Fast | More computationally expensive |
| **Deployment / Hardware** | Mobile-friendly | Often harder to run on mobile |
| **Size & Computation** | Small | Can have many computations |

so researchers asked:
Can we combine the best parts of CNNs and Transformers ?

that is basically idea behind MobileViT.


> MobileViT = CNN + Transformer, designed to be lightweight enough for mobile devices.

CNN looks nearby. Transformer looks everywhere. MobileViT tries to do both.


3. Why is it called MobileViT ?

> MobileViT= a vision Transformer designed with mobile efficiency in mind.

The architecture combines ideas from:

* CNNs
* Transformers
* MobileNet-style efficient Convolutions

The goal is:

```
High accuracy
     +
Global understanding
     +
Low computation
     +
Fast inference
     ↓
MobileViT
```


4. MobileViT builds upon ideas from MobileNet.

> "how can we make CNNs much cheaper to run ?"

One important technique is depthwise seperable convolution.

A normal convolution looks at all channels and process everything together which is expensive so depthwise seperable convultions splits the job into smaller jobs.


```
MobileNet-style:

Input
 ↓
Small operation
 ↓
Another small operation
 ↓
Output
```

Two smaller operations can be dramatically cheaper than one large operations.

This gives MobileNet its reputation for being:

> Small + fast + mobile-friendly

The MobileViT block takes an image, uses CNNs to understand local details, then uses a transformer to understand relationship  between distant areas.



5. MobileViT

Suppose we have an image:

```
H*W*C
```
where:
- H = height
- W = width
- C = number of channels

For example:

```

224 × 224 × 3

Height = 224
Width  = 224
Channels = 3

with three channels which is

Red
Green
Blue

```

* Step 1 - Convolution

The MobileViT block first uses a convolution because CNNs are excellent at detecting local pattens.

For example:

```
small region
     ↓
edges
corners
textures
shapes
```

so CNN part says let me understand small details.

* Step 2- Create patches
The image is divided into smaller pieces called patches.

```
Original image

+----+----+----+----+
| P1 | P2 | P3 | P4 |
+----+----+----+----+
| P5 | P6 | P7 | P8 |
+----+----+----+----+
| P9 |P10 |P11 |P12 |
+----+----+----+----+
```

Each little square is a patch. This is similar to Vision Transformers.

We divide image into patches because transformers doesnot directly work with image. Instead of giving whole image we give 100 little pieces of the image, then the transformer can process those pieces.

* Step 3 - Flateten the patches:

Suppose one patch looks like

```
[ 1  2 ]
[ 3  4 ]


we can fltatten it:

[1, 2, 3, 4]


```

so

> Unfolding = turning image patches into sequences that the transformer can process.


```
Image
 ↓
Cut into patches
 ↓
Flatten
 ↓
Transformer
```


* step 4- Transformer looks globally

Now the transformer gets those patches.

```
Patch 1 = dog's head
Patch 2 = dog's body
Patch 3 = dog's tail
```
Transformer can learn:
> "patch1 and Patch3 probably belong to the same object"

This is global information.

The transformer isnot only asking " what is inside the patch ? but how are these patches related to each other ? "

which is what attention about.

Attention helps the model figure out relation rather than treating every patch completely independently.

* Step 5 - Fold the patches back

After the transformer processes the patches, we need to turn them back into an image like structure.

so,

```
Patches
   ↓
Transformer
   ↓
Processed patches
   ↓
Put them back into their original locations
   ↓
Feature map
```

This is called folding or reconstruction.

The important thing is :

> We dont completely lose where each patch came from.

Patch 1 stays associated with its location.

Patch 2 stays associated with its location.

and so on



Why is location Important ?

Because some feature is different from some random feature somewhere else in picture.

MobileViT maintains spatial information while allowing the transformer to establish the global relationships.



* step 6 - Another convolution

After the transformer, MobileViT uses convolution again because CNNs are still really good at local processing.


```
CNN
 ↓
Local information
 ↓
Patches
 ↓
Transformer
 ↓
Global information
 ↓
CNN
 ↓
Local processing again

which is core idea.

```

* step 7 - Combine with original information

MobileViT combines the processed representation with the original input.

```
Original information
        +
New information
        ↓
Better representation
```


This is similar to skip/residual connection because we dont want to throw away the useful information we already had.


> Keep what we already know, and add what we learned.


```
              IMAGE
                │
                ▼
         ┌─────────────┐
         │     CNN     │
         │ local info  │
         └─────────────┘
                │
                ▼
        Divide into patches
                │
                ▼
             Flatten
                │
                ▼
        ┌─────────────┐
        │ Transformer │
        │ global info │
        └─────────────┘
                │
                ▼
              Fold
                │
                ▼
         ┌─────────────┐
         │     CNN     │
         │ local info  │
         └─────────────┘
                │
                ▼
        Combine with input
                │
                ▼
              OUTPUT
```

> CNN → unfold → Transformer → fold → CNN → combine

- Receptive field : How much of the original image can influence a particular feature.

If CNN is looking through a tiny window:

```
+---+
| 👁 |
+---+

```

it sees a only small part of image, its receptive field is small and as we add more layers, the model can indirectly seee more:

```
Layer 1 → small area
Layer 2 → larger area
Layer 3 → even larger area
```

Eventually, it might see much of the image.


MobileViT has large receptive field because transformer can connect patches across the image.

For example:

```
Patch A ───────────────┐
                       │
Patch B ────────┐      │
                ↓      ↓
              Transformer
                ↑      ↑
Patch C ────────┘      │
                       │
Patch D ───────────────┘
```

so information from one part of the image can interact with information from another part. thats why we can say MobileViT gets a receptive field covering the whole image within its Transformer processing.




### The problem with original MobileViT

Transformer uses self-attention, traditonal self-attention becomes expensive as the number of token increases. Time compelxity is approximately $O(k~2)$.

where k = number of tokens/patches


The number of tokens increased by 100×, but the computation increased by roughly 10,000×.

That's quadratic growth.


Thid id bad for phones as they have limited:

* CPU/GPU resources
* Battery
* Memory
* Heat Capacitty

so, MobileViT v2 comes which have important change:

> Seperable self-attention

Now MobileViT v2:
> $O(k)$


Traditional self attentions asks:

Imagine 4 patches:

```
A B C D

It considers relationships like this

A → B
A → C
A → D

B → A
B → C
B → D

C → A
C → B
C → D

D → A
D → B
D → C


As the no of tokens increases, the relationships explodes.

```


But,

**Seperable self-atttention takes a cheaper route**


Instead of doing pairwise interactions with huge matrix multiplications, seperable self-attention uses a more efficient computation.

> Separable self-attention reorganizes the computation so it can be done much more cheaply.


#### "batch-wise matrix multiplication"

Transformers represent information using matrices, then they perform large matrix multiplication to calculate attention.  But it can be expensive on small resource constrained devices.

So MobileViT v2 says:

> "Can we get similar useful attention behavior without these expensive matrix-multiplication operations?"

That's one of the important contributions of separable self-attention.




> CNNs naturally emphasize local interactions, while Transformers provide a more direct mechanism for modeling long-range relationships.



```
The block receives feature maps containing visual information.

First, use a CNN to understand local visual patterns.

Cut the feature map into little pieces and turn those pieces into sequences.

Let the Transformer figure out relationships between different parts of the image.

Put the pieces back into their original spatial arrangement.

Use a cheap 1×1 convolution to mix information across channels.

Combine the new information with information from before, so useful information isn't lost.

```

> 3×3 convolution → looks around
> 1×1 convolution → mixes channels

Downsampling: Making the spatial dimensions smaller.

Global Pooling: Take everything the model learned across the image and summarize it.


----


```

Traditional CNN
     ↓
Very good local understanding
     ↓
But global relationships are harder
     
Traditional Transformer
     ↓
Excellent global relationships
     ↓
But expensive for mobile

MobileViT
     ↓
CNN + Transformer
     ↓
Local + Global
     ↓
Mobile-friendly

MobileViT v2
     ↓
MobileViT
     +
Separable self-attention
     ↓
Much cheaper attention
     ↓
O(k) instead of O(k²)
     ↓
Faster mobile inference


```


MobileViT v2 — Cheat Sheet

Problem:

CNNs → fast + lightweight + good local features

Transformers → good global relationships but computationally expensive

Mobile devices need → low latency + low memory + low computation


Solution:

MobileViT = CNN + Transformer


```
MobileViT Block:


Feature map
   ↓
CNN
   ↓
Local features
   ↓
Unfold into patches
   ↓
Flatten → tokens
   ↓
Transformer
   ↓
Global relationships
   ↓
Fold patches
   ↓
Feature map
   ↓
1×1 convolution
   ↓
Combine with input



Key idea:


CNN = local
Transformer = global
MobileViT = local + global


MobileViT uses CNNs to efficiently understand local visual details and Transformers to understand relationships across the whole image; MobileViT v2 makes the Transformer part much cheaper using separable self-attention.

----

In [2]:
pip install transformers

In [9]:
from transformers import AutoImageProcessor, MobileViTV2ForImageClassification
import requests
from datasets import load_dataset
from PIL import Image

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image=Image.open(requests.get(url,stream=True).raw)


image_processor=AutoImageProcessor.from_pretrained(
    "apple/mobilevitv2-1.0-imagenet1k-256"
)


model= MobileViTV2ForImageClassification.from_pretrained(
    "apple/mobilevitv2-1.0-imagenet1k-256"
)

inputs=image_processor(image,return_tensors="pt")

logits=model(**inputs).logits

predicted_label=logits.argmax(-1).item()
print(model.config.id2label[predicted_label])



Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

tabby, tabby cat
